# CSET485 – AI and Society · Assignment 1
**Roll number:** 0683 → last digit **3** → **German Credit Data** (UCI Statlog)
**Random seed:** last three digits **683** (`random_state=683`), 70 % train / 30 % test.

Running this notebook top-to-bottom regenerates every number, table and chart in the PDF report.
All outputs are written to `results/` (`results.json`, CSV tables, PNG figures).

**Target encoding.** UCI codes `class=1` = good credit and `class=2` = bad credit.
Following the brief, `approve = 1` (good → approve) and `approve = 0` (bad → reject).
Because the positive class is *approve*, the brief's label "high risk" (score ≥ cut-off)
here means **"flagged = predicted approve"**; we keep the brief's procedure and say so explicitly.

In [1]:
import os, json, warnings
import numpy as np, pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score
from common import SEED, load_raw, decode, NUMERIC, BASELINE, CODES
warnings.filterwarnings("ignore")
os.makedirs("results", exist_ok=True)
R = {}                      # every number cited in the report is stored here
np.random.seed(SEED)
print("seed =", SEED)

seed = 683


## 0. Data loading and preprocessing

In [2]:
raw = load_raw("data/german.data")     # downloads from UCI if the file is missing
df = decode(raw)
print(df.shape)
R["n_rows"] = len(df); R["n_approve"] = int(df.approve.sum()); R["n_reject"] = int((1-df.approve).sum())

# --- Missing values ---
R["missing_total"] = int(raw.isna().sum().sum())
print("missing cells:", R["missing_total"])
print("A-code check: unknown categories after decoding:",
      int(sum(df[c].isna().sum() for c in CODES)))

# --- Outliers (1.5*IQR rule) on numeric columns ---
out = {}
for c in NUMERIC:
    q1, q3 = df[c].quantile([.25, .75]); i = q3 - q1
    out[c] = int(((df[c] < q1-1.5*i) | (df[c] > q3+1.5*i)).sum())
R["outliers_iqr"] = out
R["amount_skew_raw"] = float(df.amount.skew()); R["amount_skew_log"] = float(df.log_amount.skew())
R["amount_max"] = int(df.amount.max()); R["amount_median"] = float(df.amount.median())
print(out, "skew raw/log:", round(R["amount_skew_raw"],2), round(R["amount_skew_log"],2))

(1000, 24)
missing cells: 0
A-code check: unknown categories after decoding: 0
{'duration': 70, 'amount': 72, 'installment_rate': 0, 'residence_since': 0, 'age': 23, 'existing_credits': 6, 'people_liable': 155} skew raw/log: 1.95 0.13


In [3]:
# --- Feature matrix ---
NUM_FEATS = ["duration", "log_amount", "installment_rate", "residence_since",
             "age", "existing_credits", "people_liable"]          # amount -> log(amount)
CAT_FEATS = list(CODES.keys())

def one_hot(d):
    parts = []
    for c in CAT_FEATS:
        dm = pd.get_dummies(d[c], prefix=c, dtype=float)
        base = BASELINE.get(c, d[c].value_counts().index[0])
        dm = dm.drop(columns=[f"{c}_{base}"])          # drop reference level
        parts.append(dm)
    return pd.concat(parts, axis=1)

X_cat = one_hot(df)
idx_tr, idx_te = train_test_split(df.index, test_size=0.30, random_state=SEED, stratify=None)
scaler = StandardScaler().fit(df.loc[idx_tr, NUM_FEATS])          # fit on TRAIN only
X_num = pd.DataFrame(scaler.transform(df[NUM_FEATS]), columns=NUM_FEATS, index=df.index)
X = pd.concat([X_num, X_cat], axis=1)
y = df["approve"]
Xtr, Xte, ytr, yte = X.loc[idx_tr], X.loc[idx_te], y.loc[idx_tr], y.loc[idx_te]
R["n_features"] = X.shape[1]; R["n_train"] = len(Xtr); R["n_test"] = len(Xte)
R["train_approve_rate"] = float(ytr.mean()); R["test_approve_rate"] = float(yte.mean())
print(X.shape, Xtr.shape, Xte.shape, round(ytr.mean(),3), round(yte.mean(),3))

(1000, 48) (700, 48) (300, 48) 0.71 0.677


## Part 1 — Model A (OLS linear probability) vs Model B (logistic regression)

In [4]:
XtrC, XteC = sm.add_constant(Xtr), sm.add_constant(Xte)
modelA = sm.OLS(ytr, XtrC).fit()
modelB = sm.Logit(ytr, XtrC).fit(disp=0, maxiter=500)
pA = modelA.predict(XteC); pB = modelB.predict(XteC)

medA = float(np.median(pA))
flagA = (pA >= medA).astype(int)      # "high" if >= median prediction
flagB = (pB >= 0.5).astype(int)       # "high" if P >= 0.5
cm = confusion_matrix(flagB, flagA, labels=[0, 1])   # rows = Model B, cols = Model A
mism = int((flagA != flagB).sum())

R["p1"] = dict(
    median_A=medA, min_A=float(pA.min()), max_A=float(pA.max()),
    n_A_below0=int((pA < 0).sum()), n_A_above1=int((pA > 1).sum()),
    flagA_rate=float(flagA.mean()), flagB_rate=float(flagB.mean()),
    cm=cm.tolist(), mismatch=mism, mismatch_pct=100*mism/len(yte),
    acc_A=accuracy_score(yte, flagA), acc_B=accuracy_score(yte, flagB),
    acc_A_at05=accuracy_score(yte, (pA >= 0.5).astype(int)),
    auc_A=roc_auc_score(yte, pA), auc_B=roc_auc_score(yte, pB),
    corr_AB=float(np.corrcoef(pA, pB)[0, 1]),
    ols_r2=float(modelA.rsquared), logit_pseudo_r2=float(modelB.prsquared),
    mismatch_actual_good=int(yte[flagA != flagB].sum()),
    pB_median=float(np.median(pB)),
    # where do the mismatches sit?
    mis_A1_B0=int(((flagA == 1) & (flagB == 0)).sum()),
    mis_A0_B1=int(((flagA == 0) & (flagB == 1)).sum()),
    mis_pB_mean=float(pB[flagA != flagB].mean()),
    mis_pB_min=float(pB[flagA != flagB].min()), mis_pB_max=float(pB[flagA != flagB].max()),
)
# median threshold at the 0.5 rule would match: what B-probability does the A-median correspond to?
R["p1"]["pB_quantile_of_05"] = float((pB < 0.5).mean())
# Isolate threshold vs model-shape: cut OLS at 0.5, and cut logistic at its own median
R["p1"]["mismatch_A05_vs_B05"] = int(((pA >= 0.5).astype(int) != flagB).sum())
R["p1"]["mismatch_Amed_vs_Bmed"] = int((flagA != (pB >= np.median(pB)).astype(int)).sum())
R["p1"]["mis_actual_good_pct"] = float(yte[flagA != flagB].mean())
R["p1"]["n_test_actual_good"] = int(yte.sum())
cmdf = pd.DataFrame(cm, index=["B: low (P<0.5)", "B: high (P>=0.5)"],
                    columns=[f"A: low (<{medA:.3f})", f"A: high (>={medA:.3f})"])
cmdf.to_csv("results/p1_confusion.csv")
print(cmdf); print({k: (round(v,4) if isinstance(v,float) else v) for k,v in R["p1"].items()})

fig, ax = plt.subplots(figsize=(6.4, 4.6))
col = np.where(flagA != flagB, "#d1495b", "#7a8ca3")
ax.scatter(pA, pB, c=col, s=14, alpha=.8)
ax.axvline(medA, ls="--", c="k", lw=1); ax.axhline(0.5, ls=":", c="k", lw=1)
ax.set_xlabel("Model A (OLS) predicted value"); ax.set_ylabel("Model B (logistic) P(approve)")
ax.set_title(f"Part 1: {mism} of {len(yte)} test cases disagree (red)")
ax.text(medA+0.01, 0.03, f"A median = {medA:.3f}", fontsize=8)
fig.tight_layout(); fig.savefig("results/fig_p1_scatter.png", dpi=180); plt.close(fig)

                  A: low (<0.711)  A: high (>=0.711)
B: low (P<0.5)                 80                  0
B: high (P>=0.5)               70                150
{'median_A': 0.7115, 'min_A': -0.0576, 'max_A': 1.5043, 'n_A_below0': 1, 'n_A_above1': 41, 'flagA_rate': 0.5, 'flagB_rate': 0.7333, 'cm': [[80, 0], [70, 150]], 'mismatch': 70, 'mismatch_pct': 23.3333, 'acc_A': 0.6967, 'acc_B': 0.7167, 'acc_A_at05': 0.7233, 'auc_A': 0.7811, 'auc_B': 0.7781, 'corr_AB': 0.9373, 'ols_r2': 0.2992, 'logit_pseudo_r2': 0.2889, 'mismatch_actual_good': 38, 'pB_median': 0.8061, 'mis_A1_B0': 0, 'mis_A0_B1': 70, 'mis_pB_mean': 0.6705, 'mis_pB_min': 0.5002, 'mis_pB_max': 0.9409, 'pB_quantile_of_05': 0.2667, 'mismatch_A05_vs_B05': 8, 'mismatch_Amed_vs_Bmed': 14, 'mis_actual_good_pct': 0.5429, 'n_test_actual_good': 203}


## Part 2.1 — Simpson's paradox hunt
Split variable: **sex** (derived from `personal_status`: A92 = female, A91/A93/A94 = male; A95 does not occur).
Model: logistic regression on a compact set of key predictors (keeps the female subgroup, n = 310,
well-identified) fitted on (1) the full 1,000 rows, (2) women, (3) men. Key predictor: **age**.
A scan of all numeric predictors × {sex, age group, foreign worker} is shown first so the choice is transparent.

In [5]:
d = df.copy()
d["chk_neg"] = (d.checking == "lt0DM").astype(int); d["chk_low"] = (d.checking == "0to200DM").astype(int)
d["chk_hi"] = (d.checking == "ge200DM").astype(int)
d["hist_crit"] = (d.credit_history == "critical_other_credits").astype(int)
d["sav_low"] = (d.savings == "lt100").astype(int)
d["female"] = (d.sex == "female").astype(int)
F21 = ("approve ~ duration + log_amount + age + installment_rate + existing_credits"
       " + chk_neg + chk_low + chk_hi + hist_crit + sav_low")
KEYS = ["age", "duration", "installment_rate", "existing_credits", "sav_low", "chk_neg"]

scan = []
for s, (a, b) in {"sex": ("female", "male"), "age_group": ("young_le25", "older_gt25"),
                  "foreign_worker": ("yes", "no")}.items():
    try:
        ms = [smf.logit(F21, x).fit(disp=0, maxiter=200) for x in (d, d[d[s] == a], d[d[s] == b])]
    except Exception as e:
        continue
    for k in KEYS:
        v = [m.params[k] for m in ms]
        scan.append(dict(split=s, predictor=k, full=v[0], A=v[1], B=v[2],
                         sign_reversal=len(set(np.sign(v))) > 1))
pd.DataFrame(scan).round(3).to_csv("results/p21_scan.csv", index=False)

fits = {"Full (n=1000)": smf.logit(F21, d).fit(disp=0),
        "A: Female (n=%d)" % (d.female == 1).sum(): smf.logit(F21, d[d.female == 1]).fit(disp=0),
        "B: Male (n=%d)" % (d.female == 0).sum(): smf.logit(F21, d[d.female == 0]).fit(disp=0)}
rows = []
for name, m in fits.items():
    ci = m.conf_int()
    for k in KEYS:
        rows.append(dict(model=name, predictor=k, coef=m.params[k], ci_low=ci.loc[k, 0],
                         ci_high=ci.loc[k, 1], p=m.pvalues[k], OR=np.exp(m.params[k])))
t21 = pd.DataFrame(rows); t21.round(4).to_csv("results/p21_coefficients.csv", index=False)
print(t21[t21.predictor == "age"].round(4))

# formal test of the difference: age x female interaction
mi = smf.logit(F21 + " + female + age:female", d).fit(disp=0)
ci_i = mi.conf_int().loc["age:female"]
# unadjusted (crude) patterns
crude = smf.logit("approve ~ age", d).fit(disp=0)
R["p21"] = dict(
    table=t21.round(5).to_dict("records"),
    interaction=dict(coef=float(mi.params["age:female"]), p=float(mi.pvalues["age:female"]),
                     ci=[float(ci_i[0]), float(ci_i[1])]),
    crude_age_coef=float(crude.params["age"]), crude_age_p=float(crude.pvalues["age"]),
    approve_rate_female=float(d[d.female == 1].approve.mean()),
    approve_rate_male=float(d[d.female == 0].approve.mean()),
    mean_age_female=float(d[d.female == 1].age.mean()), mean_age_male=float(d[d.female == 0].age.mean()),
    pct_young_female=float((d[d.female == 1].age <= 25).mean()),
    pct_young_male=float((d[d.female == 0].age <= 25).mean()),
    n_female=int(d.female.sum()), n_male=int((1-d.female).sum()),
)
# approval by age band within each sex (descriptive, explains the slope difference)
d["age_band"] = pd.cut(d.age, [18, 25, 35, 50, 80], labels=["19-25", "26-35", "36-50", "51+"])
band = d.pivot_table(index="age_band", columns="sex", values="approve", aggfunc="mean", observed=True)
cnt = d.pivot_table(index="age_band", columns="sex", values="approve", aggfunc="size", observed=True)
band.round(3).to_csv("results/p21_age_band_rates.csv"); R["p21"]["age_band"] = band.round(3).to_dict()
R["p21"]["age_band_n"] = cnt.to_dict()
print(band.round(3)); print(cnt); print(R["p21"]["interaction"])

# figure: predicted approval vs age, other covariates at their means
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ages = np.arange(19, 76)
for (name, m), c in zip(fits.items(), ["#333333", "#d1495b", "#2e86ab"]):
    sub = d if name.startswith("Full") else d[d.female == (1 if "Female" in name else 0)]
    g = pd.DataFrame({k: [sub[k].mean()] * len(ages) for k in m.params.index if k != "Intercept"})
    g["age"] = ages
    ax.plot(ages, m.predict(g), c=c, lw=2, label=f"{name}: β_age={m.params['age']:+.3f}")
ax.set_xlabel("Age"); ax.set_ylabel("Predicted P(approve)"); ax.legend(fontsize=8)
ax.set_title("Part 2.1: age effect differs by sex"); fig.tight_layout()
fig.savefig("results/fig_p21_age_by_sex.png", dpi=180); plt.close(fig)

                model predictor    coef  ci_low  ci_high       p      OR
0       Full (n=1000)       age  0.0140 -0.0001   0.0280  0.0518  1.0141
6   A: Female (n=310)       age  0.0413  0.0147   0.0679  0.0023  1.0422
12    B: Male (n=690)       age -0.0028 -0.0203   0.0146  0.7519  0.9972
sex       female   male
age_band               
19-25      0.552  0.612
26-35      0.615  0.740
36-50      0.811  0.752
51+        0.800  0.692
sex       female  male
age_band              
19-25        105    85
26-35        117   281
36-50         53   246
51+           35    78
{'coef': 0.040854365560288985, 'p': 0.008844108690204354, 'ci': [0.010269029296978642, 0.07143970182359932]}


## Part 2.2 — Omitted-variable bias
Full model = logistic regression on all 20 attributes (all 1,000 rows, unscaled numerics so coefficients are in natural units).
Dropped confounder: **duration** (loan length in months). Duration drives both the loan amount (r = 0.64 with log-amount)
and repayment risk, so it is a textbook confounder of the amount → approval relationship.

In [6]:
CAT_TERMS = " + ".join(f'C({c}, Treatment("{BASELINE.get(c, d[c].value_counts().index[0])}"))'
                       for c in CAT_FEATS)
F_FULL = "approve ~ duration + log_amount + installment_rate + residence_since + age + existing_credits + people_liable + " + CAT_TERMS
F_RED = F_FULL.replace("duration + ", "")
mfull = smf.logit(F_FULL, d).fit(disp=0, maxiter=500)
mred = smf.logit(F_RED, d).fit(disp=0, maxiter=500)
common_ = [c for c in mred.params.index if c != "Intercept"]
t22 = pd.DataFrame({"full": mfull.params[common_], "reduced": mred.params[common_],
                    "p_full": mfull.pvalues[common_]})
t22["abs_change"] = t22.reduced - t22.full
t22["pct_chg"] = 100 * t22.abs_change / t22.full.abs()
t22.round(4).to_csv("results/p22_all_coefficients.csv")
FOCUS = ["log_amount", "installment_rate", "age", "existing_credits",
         'C(checking, Treatment("no_account"))[T.lt0DM]',
         'C(credit_history, Treatment("existing_paid"))[T.critical_other_credits]',
         'C(savings, Treatment("lt100"))[T.ge1000]']
focus = t22.loc[FOCUS]
print(focus.round(4))
# biggest absolute shift among predictors that are meaningful (|coef| >= 0.1 in full model)
meaningful = t22[t22.full.abs() >= 0.1]
R["p22"] = dict(
    dropped="duration", corr_duration_logamount=float(d[["duration", "log_amount"]].corr().iloc[0, 1]),
    corr_duration_amount=float(d[["duration", "amount"]].corr().iloc[0, 1]),
    focus=focus.round(5).reset_index().rename(columns={"index": "term"}).to_dict("records"),
    largest_pct_meaningful=meaningful.pct_chg.abs().idxmax(),
    largest_pct_meaningful_val=float(meaningful.loc[meaningful.pct_chg.abs().idxmax(), "pct_chg"]),
    largest_pct_any=t22.pct_chg.abs().idxmax(),
    largest_pct_any_val=float(t22.loc[t22.pct_chg.abs().idxmax(), "pct_chg"]),
    largest_pct_any_full=float(t22.loc[t22.pct_chg.abs().idxmax(), "full"]),
    largest_abs_shift=t22.abs_change.abs().idxmax(),
    largest_abs_shift_val=float(t22.loc[t22.abs_change.abs().idxmax(), "abs_change"]),
    p_logamount_full=float(mfull.pvalues["log_amount"]), p_logamount_red=float(mred.pvalues["log_amount"]),
    ci_logamount_full=[float(x) for x in mfull.conf_int().loc["log_amount"]],
    ci_logamount_red=[float(x) for x in mred.conf_int().loc["log_amount"]],
    duration_coef=float(mfull.params["duration"]), duration_p=float(mfull.pvalues["duration"]),
    aic_full=float(mfull.aic), aic_red=float(mred.aic),
    pseudo_r2_full=float(mfull.prsquared), pseudo_r2_red=float(mred.prsquared),
    n_sign_flips=int((np.sign(t22.full) != np.sign(t22.reduced)).sum()),
    n_meaningful_over20=int((meaningful.pct_chg.abs() > 20).sum()), n_meaningful=int(len(meaningful)),
)
# auxiliary regression: how much of duration is "explained" by log_amount (OVB formula ingredient)
aux = smf.ols("duration ~ log_amount", d).fit()
R["p22"]["aux_slope_duration_on_logamount"] = float(aux.params["log_amount"])
R["p22"]["ovb_approx"] = float(mfull.params["duration"] * aux.params["log_amount"])
print({k: v for k, v in R["p22"].items() if k != "focus"})

                                                      full  reduced  p_full  \
log_amount                                         -0.2255  -0.6289  0.2054   
installment_rate                                   -0.2765  -0.3724  0.0021   
age                                                 0.0145   0.0166  0.1143   
existing_credits                                   -0.2773  -0.2315  0.1418   
C(checking, Treatment("no_account"))[T.lt0DM]      -1.6894  -1.7188  0.0000   
C(credit_history, Treatment("existing_paid"))[T...  0.8560   0.8537  0.0009   
C(savings, Treatment("lt100"))[T.ge1000]            1.3098   1.3443  0.0117   

                                                    abs_change   pct_chg  
log_amount                                             -0.4035 -178.9610  
installment_rate                                       -0.0960  -34.7089  
age                                                     0.0021   14.7998  
existing_credits                                        0.0458   16

## Part 2.3 — Adversarial subset (confident, wrong predictions)
Model B (fitted on the training set) applied to the 300-row test set.

In [7]:
te = df.loc[idx_te].copy(); te["p"] = pB.values if hasattr(pB, "values") else pB
te["fp_conf"] = (te.p >= 0.75) & (te.approve == 0)
te["fn_conf"] = (te.p <= 0.25) & (te.approve == 1)
te["adv"] = te.fp_conf | te.fn_conf
adv_all = te[te.adv]
adv = adv_all if len(adv_all) <= 50 else adv_all.sample(50, random_state=SEED)

def profile(x):
    return dict(n=len(x), age_mean=float(x.age.mean()), age_median=float(x.age.median()),
                pct_young=float((x.age <= 25).mean()), pct_female=float((x.sex == "female").mean()),
                pct_foreign=float((x.foreign_worker == "yes").mean()),
                pct_no_checking=float((x.checking == "no_account").mean()),
                pct_crit_history=float((x.credit_history == "critical_other_credits").mean()),
                pct_dur_le24=float((x.duration <= 24).mean()),
                duration_mean=float(x.duration.mean()), amount_median=float(x.amount.median()),
                pct_own_house=float((x.housing == "own").mean()),
                pct_sav_low=float((x.savings == "lt100").mean()),
                pct_actual_bad=float((x.approve == 0).mean()), p_mean=float(x.p.mean()))

R["p23"] = dict(n_fp=int(te.fp_conf.sum()), n_fn=int(te.fn_conf.sum()), n_total=int(te.adv.sum()),
                pct_of_test=100 * te.adv.mean(), n_used=len(adv),
                n_pred_ge075=int((te.p >= .75).sum()), n_pred_le025=int((te.p <= .25).sum()),
                adv=profile(adv), test=profile(te))
# conditional error rate inside the "confident approve" band for the trait
band = te[te.p >= 0.75]
R["p23"]["fp_rate_in_band"] = float((band.approve == 0).mean())
for col, val in [("checking", "no_account"), ("credit_history", "critical_other_credits")]:
    sub = df.loc[idx_tr]
    R["p23"][f"train_bad_rate_{val}"] = float(1 - sub[sub[col] == val].approve.mean())
    R["p23"][f"train_n_{val}"] = int((sub[col] == val).sum())
R["p23"]["train_bad_rate_overall"] = float(1 - df.loc[idx_tr].approve.mean())
# coefficients of the two driving dummies (logit units, relative to baseline levels)
cb = modelB.params
R["p23"]["coef_checking_lt0"] = float(cb["checking_lt0DM"])
R["p23"]["coef_checking_0to200"] = float(cb["checking_0to200DM"])
R["p23"]["coef_crit_hist"] = float(cb["credit_history_critical_other_credits"])
R["p23"]["coef_duration_std"] = float(cb["duration"])
R["p23"]["duration_sd"] = float(scaler.scale_[0])
# Within "no checking account" applicants in TEST: bad rate vs mean predicted P
nc = te[te.checking == "no_account"]
R["p23"]["test_nochk_n"] = len(nc); R["p23"]["test_nochk_bad_rate"] = float(1 - nc.approve.mean())
R["p23"]["test_nochk_mean_p"] = float(nc.p.mean())
# interaction check: does adding checking x duration interaction (train) help?
dtr = d.loc[idx_tr].copy(); dte = d.loc[idx_te].copy(); dtr["nochk"] = (dtr.checking == "no_account").astype(int)
dte["nochk"] = (dte.checking == "no_account").astype(int)
mi1 = smf.logit("approve ~ nochk + duration + log_amount + hist_crit", dtr).fit(disp=0)
mi2 = smf.logit("approve ~ nochk * duration + log_amount + hist_crit", dtr).fit(disp=0)
R["p23"]["lr_test_interaction_p"] = float(1 - __import__("scipy").stats.chi2.cdf(2*(mi2.llf - mi1.llf), 1))
R["p23"]["interaction_coef"] = float(mi2.params["nochk:duration"])
# --- What do these rows share? Decompose each row's logit into per-variable contributions ---
contrib = Xte.loc[adv.index] * modelB.params.drop("const")
groups = {c: [k for k in contrib.columns if k == c or k.startswith(c + "_")] for c in NUM_FEATS + CAT_FEATS}
gcon = pd.DataFrame({g: contrib[cols_].sum(axis=1) for g, cols_ in groups.items()})
# direction toward the (wrong) prediction: + for false positives, - for false negatives
sign = np.where(adv.approve == 0, 1, -1)
toward = gcon.mul(sign, axis=0)
top_var = toward.idxmax(axis=1)
R["p23"]["top_misleading_var_counts"] = top_var.value_counts().to_dict()
R["p23"]["mean_push_toward_wrong"] = toward.mean().sort_values(ascending=False).round(3).head(6).to_dict()
# Shared trait: the strongest predictor (checking status) points to the WRONG class
fav_chk = adv.checking.isin(["no_account", "ge200DM"])
trait = ((adv.approve == 0) & fav_chk) | ((adv.approve == 1) & ~fav_chk)
R["p23"]["trait_checking_misleads_n"] = int(trait.sum()); R["p23"]["trait_checking_misleads_pct"] = float(trait.mean())
fp_rows, fn_rows = adv[adv.approve == 0], adv[adv.approve == 1]
R["p23"]["fp_pct_dur_le24"] = float((fp_rows.duration <= 24).mean())
R["p23"]["fp_pct_fav_checking"] = float(fp_rows.checking.isin(["no_account", "ge200DM"]).mean())
R["p23"]["fp_pct_nochk"] = float((fp_rows.checking == "no_account").mean())
R["p23"]["fn_pct_unfav_checking"] = float((~fn_rows.checking.isin(["no_account", "ge200DM"])).mean())
R["p23"]["fn_pct_short_emp"] = float(fn_rows.employment.isin(["unemployed", "lt1y"]).mean())
R["p23"]["fn_mean_duration"] = float(fn_rows.duration.mean()); R["p23"]["fp_mean_duration"] = float(fp_rows.duration.mean())
R["p23"]["fp_mean_p"] = float(fp_rows.p.mean()); R["p23"]["fn_mean_p"] = float(fn_rows.p.mean())
# 'Trait' = row's two most influential signals (checking + duration) both point to the wrong class
fp_trait = fp_rows.checking.isin(["no_account", "ge200DM"]) | (fp_rows.duration <= 12)
fn_trait = (~fn_rows.checking.isin(["no_account", "ge200DM"])) | (fn_rows.duration >= 30)
R["p23"]["any_key_signal_misleads_pct"] = float((fp_trait.sum() + fn_trait.sum()) / len(adv))
# same trait prevalence in the full test set (signal points the other way from the truth)
tfav = te.checking.isin(["no_account", "ge200DM"])
t_trait = ((te.approve == 0) & (tfav | (te.duration <= 12))) | ((te.approve == 1) & (~tfav | (te.duration >= 30)))
R["p23"]["any_key_signal_misleads_pct_test"] = float(t_trait.mean())
t_trait1 = ((te.approve == 0) & tfav) | ((te.approve == 1) & ~tfav)
R["p23"]["trait_checking_misleads_pct_test"] = float(t_trait1.mean())
print(top_var.value_counts()); print(toward.mean().sort_values(ascending=False).round(3).head(6))

cols = ["checking", "credit_history", "duration", "amount", "age", "sex", "savings",
        "purpose", "employment", "housing", "approve", "p"]
adv[cols].sort_values("p").to_csv("results/p23_adversarial_rows.csv")
comp = pd.DataFrame({"Adversarial subset": profile(adv), "Full test set": profile(te)}).T
comp.round(3).to_csv("results/p23_profile.csv"); print(comp.round(3).T)
print({k: v for k, v in R["p23"].items() if k not in ("adv", "test")})

checking            7
savings             7
credit_history      5
foreign_worker      3
purpose             2
other_debtors       2
installment_rate    2
duration            1
employment          1
job                 1
Name: count, dtype: int64
savings           0.237
foreign_worker    0.235
other_debtors     0.201
duration          0.184
credit_history    0.141
purpose           0.095
dtype: float64
                  Adversarial subset  Full test set
n                             31.000        300.000
age_mean                      35.258         35.130
age_median                    33.000         33.000
pct_young                      0.226          0.207
pct_female                     0.258          0.303
pct_foreign                    0.903          0.940
pct_no_checking                0.387          0.380
pct_crit_history               0.290          0.313
pct_dur_le24                   0.774          0.753
duration_mean                 20.935         21.363
amount_median          

## Part 2.4 — Fairness trade-off sweep
Sensitive attribute: **age group**. Group A = young (≤ 25), Group B = older (> 25) — the standard protected split for this dataset
(age is a protected characteristic under EU anti-discrimination law). Evaluated on the 300-row test set, Ŷ = 1 means *approve*.

In [8]:
gA = te.age <= 25; gB = ~gA
R["p24"] = dict(nA=int(gA.sum()), nB=int(gB.sum()),
                base_A=float(te[gA].approve.mean()), base_B=float(te[gB].approve.mean()))

def rates(yt, yp):
    tp = ((yp == 1) & (yt == 1)).sum(); fp = ((yp == 1) & (yt == 0)).sum()
    pos = (yt == 1).sum(); neg = (yt == 0).sum()
    return dict(sel=yp.mean(), tpr=tp / pos if pos else np.nan, fpr=fp / neg if neg else np.nan,
                ppv=tp / (tp + fp) if (tp + fp) else np.nan)

ths = np.round(np.arange(0.05, 0.96, 0.10), 2)
rows = []
for t in ths:
    yp = (te.p >= t).astype(int)
    a, b = rates(te.approve[gA], yp[gA]), rates(te.approve[gB], yp[gB])
    dp = a["sel"] / b["sel"] if b["sel"] else np.nan
    tr = a["tpr"] / b["tpr"] if b["tpr"] else np.nan
    fr = a["fpr"] / b["fpr"] if b["fpr"] else np.nan
    eo = np.nanmin([tr, fr]) if not (np.isnan(tr) and np.isnan(fr)) else np.nan
    pp = a["ppv"] / b["ppv"] if (b["ppv"] and not np.isnan(a["ppv"])) else np.nan
    rows.append(dict(threshold=t, DP=dp, EO=eo, PP=pp, TPR_ratio=tr, FPR_ratio=fr,
                     sel_A=a["sel"], sel_B=b["sel"], TPR_A=a["tpr"], TPR_B=b["tpr"],
                     FPR_A=a["fpr"], FPR_B=b["fpr"], PPV_A=a["ppv"], PPV_B=b["ppv"],
                     accuracy=accuracy_score(te.approve, yp), approve_rate=yp.mean()))
fair = pd.DataFrame(rows)
fair.round(4).to_csv("results/p24_fairness_sweep.csv", index=False)
R["p24"]["sweep"] = fair.round(5).replace({np.nan: None}).to_dict("records")
print(fair[["threshold", "DP", "EO", "PP", "accuracy", "sel_A", "sel_B", "TPR_A", "TPR_B", "FPR_A", "FPR_B", "PPV_A", "PPV_B"]].round(3).to_string())

fig, ax = plt.subplots(figsize=(6.6, 5.6))
ax.add_patch(plt.Rectangle((0.8, 0.8), 0.45, 0.45, color="#9bc53d", alpha=.15, lw=0))
ax.text(0.81, 1.21, "four-fifths band (0.8–1.25)", fontsize=7, color="#4a7a12")
ok = fair.dropna(subset=["DP", "EO"])
ax.plot(ok.DP, ok.EO, "-", c="#bbbbbb", lw=1, zorder=1)
sc = ax.scatter(ok.DP, ok.EO, c=ok.threshold, cmap="viridis", s=60, zorder=2, edgecolor="k", lw=.5)
OFF = {0.05: (10, 6), 0.15: (10, -4), 0.25: (-52, -6), 0.35: (10, -12), 0.55: (8, 4), 0.65: (-52, -4)}
for _, r in ok.iterrows():
    dx, dy = OFF.get(round(r.threshold, 2), (6, 4))
    ax.annotate(f"t={r.threshold:.2f}", (r.DP, r.EO), textcoords="offset points", xytext=(dx, dy), fontsize=8,
                arrowprops=dict(arrowstyle="-", lw=.4, color="#888") if abs(dx) > 20 else None)
ax.plot([1], [1], marker="*", c="#d1495b", ms=14, zorder=0); ax.text(1.06, 1.08, "ideal (1, 1)", color="#d1495b", fontsize=8)
ax.set_xlim(0, 1.5); ax.set_ylim(0, 1.5)
ax.set_xlabel("Demographic parity ratio  P(Ŷ=1|young)/P(Ŷ=1|older)")
ax.set_ylabel("Equalized odds ratio  min(TPR ratio, FPR ratio)")
ax.set_title("Part 2.4: fairness trade-off across thresholds (test set)")
plt.colorbar(sc, label="threshold"); fig.tight_layout()
fig.savefig("results/fig_p24_tradeoff.png", dpi=180); plt.close(fig)

fig, ax = plt.subplots(figsize=(6.6, 3.8))
for m, c in [("DP", "#2e86ab"), ("EO", "#d1495b"), ("PP", "#f0a202")]:
    ax.plot(fair.threshold, fair[m], "o-", c=c, label=m)
ax.plot(fair.threshold, fair.accuracy, "s--", c="#555", label="accuracy")
ax.axhline(1, c="k", lw=.7, ls=":"); ax.set_xlabel("threshold t"); ax.set_ylabel("ratio (young / older)")
ax.legend(fontsize=8, ncol=4); ax.set_title("Part 2.4: DP, EO, PP and accuracy vs threshold")
fig.tight_layout(); fig.savefig("results/fig_p24_metrics.png", dpi=180); plt.close(fig)

   threshold     DP     EO     PP  accuracy  sel_A  sel_B  TPR_A  TPR_B  FPR_A  FPR_B  PPV_A  PPV_B
0       0.05  1.004  1.000  0.801     0.673  1.000  0.996  1.000  0.994  1.000  1.000  0.565  0.705
1       0.15  0.960  0.943  0.819     0.690  0.935  0.975  0.971  0.988  0.889  0.943  0.586  0.716
2       0.25  0.908  0.907  0.799     0.700  0.855  0.941  0.886  0.976  0.815  0.857  0.585  0.732
3       0.35  0.927  0.911  0.786     0.720  0.806  0.870  0.857  0.940  0.741  0.700  0.600  0.763
4       0.45  0.792  0.822  0.851     0.723  0.629  0.794  0.743  0.881  0.481  0.586  0.667  0.783
5       0.55  0.716  0.648  0.913     0.723  0.532  0.744  0.686  0.839  0.333  0.514  0.727  0.797
6       0.65  0.700  0.648  0.921     0.717  0.468  0.668  0.629  0.780  0.259  0.400  0.759  0.824
7       0.75  0.564  0.389  0.997     0.717  0.339  0.601  0.514  0.732  0.111  0.286  0.857  0.860
8       0.85  0.629  0.185  1.077     0.673  0.306  0.487  0.514  0.607  0.037  0.200  0.947  0.879


In [9]:
# Quantify the trade-off between consecutive thresholds where DP improves (moves toward 1) but EO worsens.
tr_rows = []
for i in range(len(fair)):
    for j in range(len(fair)):
        if i == j: continue
        a, b = fair.iloc[i], fair.iloc[j]
        if any(pd.isna([a.DP, b.DP, a.EO, b.EO])): continue
        dp_gain = abs(a.DP - 1) - abs(b.DP - 1)       # >0: b closer to parity than a
        eo_loss = abs(b.EO - 1) - abs(a.EO - 1)       # >0: b further from parity than a
        if dp_gain > 0 and eo_loss > 0:
            tr_rows.append(dict(t_from=a.threshold, t_to=b.threshold, DP_from=a.DP, DP_to=b.DP,
                                EO_from=a.EO, EO_to=b.EO,
                                DP_gain_pct=100 * (b.DP - a.DP) / a.DP, EO_loss_pct=100 * (a.EO - b.EO) / a.EO,
                                acc_from=a.accuracy, acc_to=b.accuracy))
trade = pd.DataFrame(tr_rows)
trade["cost"] = trade.EO_loss_pct / trade.DP_gain_pct.abs()
trade.round(4).to_csv("results/p24_tradeoff_pairs.csv", index=False)
R["p24"]["tradeoff_pairs"] = trade.round(5).to_dict("records")
print(trade.round(3).to_string())

# DP-vs-PP conflict: lowering the threshold from 0.75 to 0.35 moves DP toward 1 but pushes PP away from 1
def pair(t1, t2):
    a = fair[np.isclose(fair.threshold, t1)].iloc[0]; b = fair[np.isclose(fair.threshold, t2)].iloc[0]
    return {m: dict(frm=float(a[m]), to=float(b[m]), pct=float(100 * (b[m] - a[m]) / a[m]))
            for m in ["DP", "EO", "PP", "accuracy", "sel_A", "sel_B", "FPR_A", "FPR_B", "TPR_A", "TPR_B", "PPV_A", "PPV_B"]}
R["p24"]["pair_075_035"] = pair(0.75, 0.35)
R["p24"]["pair_055_035"] = pair(0.55, 0.35)
R["p24"]["pair_035_045"] = pair(0.35, 0.45)
for k in ["pair_075_035", "pair_055_035"]:
    v = R["p24"][k]; print(k, {m: (round(x['frm'],3), round(x['to'],3), round(x['pct'],1)) for m, x in v.items()})
# no threshold satisfies all three within the four-fifths band
inband = fair[(fair.DP.between(.8, 1.25)) & (fair.EO.between(.8, 1.25)) & (fair.PP.between(.8, 1.25))]
R["p24"]["thresholds_all_three_in_band"] = inband.threshold.tolist()
R["p24"]["counts_A_test"] = dict(n=int(gA.sum()), pos=int(te[gA].approve.sum()), neg=int((1 - te[gA].approve).sum()))
R["p24"]["counts_B_test"] = dict(n=int(gB.sum()), pos=int(te[gB].approve.sum()), neg=int((1 - te[gB].approve).sum()))
print("all three in [0.8,1.25]:", R["p24"]["thresholds_all_three_in_band"], R["p24"]["counts_A_test"], R["p24"]["counts_B_test"])

   t_from  t_to  DP_from  DP_to  EO_from  EO_to  DP_gain_pct  EO_loss_pct  acc_from  acc_to   cost
0    0.75  0.85    0.564  0.629    0.389  0.185       11.535       52.381     0.717   0.673  4.541
pair_075_035 {'DP': (0.564, 0.927, 64.5), 'EO': (0.389, 0.911, 134.4), 'PP': (0.997, 0.786, -21.1), 'accuracy': (0.717, 0.72, 0.5), 'sel_A': (0.339, 0.806, 138.1), 'sel_B': (0.601, 0.87, 44.8), 'FPR_A': (0.111, 0.741, 566.7), 'FPR_B': (0.286, 0.7, 145.0), 'TPR_A': (0.514, 0.857, 66.7), 'TPR_B': (0.732, 0.94, 28.5), 'PPV_A': (0.857, 0.6, -30.0), 'PPV_B': (0.86, 0.763, -11.3)}
pair_055_035 {'DP': (0.716, 0.927, 29.6), 'EO': (0.648, 0.911, 40.6), 'PP': (0.913, 0.786, -13.9), 'accuracy': (0.723, 0.72, -0.5), 'sel_A': (0.532, 0.806, 51.5), 'sel_B': (0.744, 0.87, 16.9), 'FPR_A': (0.333, 0.741, 122.2), 'FPR_B': (0.514, 0.7, 36.1), 'TPR_A': (0.686, 0.857, 25.0), 'TPR_B': (0.839, 0.94, 12.1), 'PPV_A': (0.727, 0.6, -17.5), 'PPV_B': (0.797, 0.763, -4.2)}
all three in [0.8,1.25]: [0.05, 0.15] {'n': 62, 

In [10]:
with open("results/results.json", "w") as f:
    json.dump(R, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("saved results/results.json")

saved results/results.json
